# Computational Thinking: Problems, Algorithms, Programs

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - The core concepts of computational thinking as a problem-solving framework
> - Decomposition — breaking complex problems into smaller, manageable parts
> - Pattern recognition — identifying repetition and regularity in a problem
> - Abstraction — hiding away details that are not relevant to the task at hand
> - Logic — establishing and checking facts to reason about a solution
> - Algorithms — developing systematic, step-by-step solutions with predictable outcomes
> - Pseudocode — outlining logic and structure in plain language before coding
> - Programming/automation — translating a specification into machine instructions
> - Validation — using facts and patterns to evaluate whether an outcome is correct
> - The Turtle graphics library and the Google Colab / Jupyter notebook environment
> - Using GenAI as an iterative tool (e.g. recreating an optical illusion) and its limitations

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ------------------------------------------------------------- drawing ---

INK = "#1a1a1a"

MUTED = "#6b7280"

FILL = "#eef2f7"

ACCENT = "#2563eb"

WARM = "#b45309"

EDGE = "#334155"

def _frame(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=MUTED, pad=8)

def _box(ax, xy, text, w=2.6, h=0.9, fc=FILL, ec=EDGE, fs=9, bold=False):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
            zorder=3, fontweight="bold" if bold else "normal")
    return xy

def _diamond(ax, xy, text, w=3.0, h=1.5, fc="#fff7ed", ec=WARM, fs=9):
    x, y = xy
    ax.add_patch(Polygon(
        [(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
        closed=True, linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK, zorder=3)
    return xy

def _dot(ax, xy, r=0.09):
    ax.add_patch(Circle(xy, r, facecolor=EDGE, edgecolor=EDGE, zorder=4))
    return xy

def _arrow(ax, pts, label=None, label_at=0.5, label_off=(0.0, 0.18),
           color=EDGE, ha="center"):
    """Poly-line arrow through `pts` (elbow routing), head on the last segment."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    ax.add_line(Line2D(xs[:-1] + [xs[-1]], ys[:-1] + [ys[-1]],
                       color=color, linewidth=1.3, zorder=1,
                       solid_capstyle="round"))
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=color, linewidth=1.3,
                                shrinkA=0, shrinkB=0), zorder=1)
    if label:
        i = max(0, min(len(pts) - 2, int(label_at * (len(pts) - 1))))
        mx = (pts[i][0] + pts[i + 1][0]) / 2 + label_off[0]
        my = (pts[i][1] + pts[i + 1][1]) / 2 + label_off[1]
        ax.text(mx, my, label, fontsize=8, color=MUTED, ha=ha, va="center")

def _line(ax, pts, color=EDGE):
    """Poly-line with no arrowhead — for merging branches into a shared rail."""
    ax.add_line(Line2D([p[0] for p in pts], [p[1] for p in pts], color=color,
                       linewidth=1.3, zorder=1, solid_capstyle="round"))

def _cellgrid(ax, values, origin=(0, 0), cw=1.0, ch=1.0, fs=13, fc="white"):
    """A row/table of boxed cells; `values` is a list of rows."""
    x0, y0 = origin
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            x = x0 + c * cw
            y = y0 - r * ch
            ax.add_patch(plt.Rectangle((x, y - ch), cw, ch, facecolor=fc,
                                       edgecolor=EDGE, linewidth=1.2, zorder=2))
            ax.text(x + cw / 2, y - ch / 2, str(v), ha="center", va="center",
                    fontsize=fs, color=INK, family="monospace", zorder=3)

def _mockwindow(ax, w, h, title, body, titlebar="#d7dde5", face="#ffffff",
                fs=9, textcolor=INK):
    """A framed window with a title bar and monospaced body lines."""
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=face, edgecolor=EDGE,
                               linewidth=1.2, zorder=1))
    ax.add_patch(plt.Rectangle((0, h - 0.55), w, 0.55, facecolor=titlebar,
                               edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax.text(0.2, h - 0.28, title, fontsize=9, va="center", color=INK, zorder=3)
    y = h - 1.05
    for line, colour in body:
        ax.text(0.25, y, line, fontsize=fs, va="center", family="monospace",
                color=colour or textcolor, zorder=3)
        y -= 0.5

def _index_grid(ax, items, top_label, side_label, fs=13):
    n = len(items)
    _cellgrid(ax, [items], origin=(0, 1), fs=fs)
    for i in range(n):
        ax.text(i + 0.5, 1.25, str(i), ha="center", va="bottom", fontsize=10,
                color=ACCENT)
    if top_label:
        ax.text(-0.25, 1.3, top_label, ha="right", va="bottom", fontsize=9,
                color=ACCENT)
    if side_label:
        ax.text(-0.25, 0.5, side_label, ha="right", va="center", fontsize=9,
                color=ACCENT)
    _frame(ax, (-6.4, n + 0.4), (-0.4, 2.0))

def _double_diamond(ax, stage=None):
    names = ["Understand", "Design", "Implement", "Evaluate"]
    # two diamonds: centres at x=2.6 and x=7.8, half-width 2.6, half-height 2.0
    for d, cx in enumerate((2.6, 7.8)):
        left, right, top, bot = cx - 2.6, cx + 2.6, 2.0, -2.0
        for half in (0, 1):
            name = names[2 * d + half]
            tri = ([(left, 0), (cx, top), (cx, bot)] if half == 0
                   else [(cx, top), (right, 0), (cx, bot)])
            on = (name == stage)
            ax.add_patch(Polygon(tri, closed=True, zorder=1,
                                 facecolor="#b9bfc7" if on else "#eceef1",
                                 edgecolor="none"))
            tx = cx - 1.3 if half == 0 else cx + 1.3
            ax.text(tx, 0, name, ha="center", va="center", fontsize=10,
                    color=INK if on else MUTED,
                    fontweight="bold" if on else "normal", zorder=3)
        ax.add_patch(Polygon([(left, 0), (cx, top), (right, 0), (cx, bot)],
                             closed=True, facecolor="none", edgecolor=INK,
                             linewidth=2.2, zorder=2))
        ax.plot([cx, cx], [top, bot], color=INK, linewidth=1.0, zorder=2)
    for x, y in ((0.0, 0.0), (5.2, 0.0), (10.4, 0.0)):
        ax.add_patch(Circle((x, y), 0.22, facecolor="#c9ced6", edgecolor=INK,
                            linewidth=1.2, zorder=4))
    ax.plot([-1.5, -0.22], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([10.62, 11.9], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([5.2, 5.2], [-0.22, -3.1], color=INK, linewidth=1.6, zorder=2)
    ax.text(-1.7, 0, "Problem", ha="right", va="center", fontsize=10)
    ax.text(12.1, 0, "Program", ha="left", va="center", fontsize=10)
    ax.text(5.2, -3.35, "Specification", ha="center", va="top", fontsize=10)
    _frame(ax, (-4.2, 14.2), (-4.2, 2.6))

def draw_lego_instructions(ax):
    """Stands in for lego.jpg — a photo of a LEGO instruction sheet.

    Stylized, not the original photograph (see FIGURES.md).
    """
    def brick(x, y, studs, colour):
        ax.add_patch(plt.Rectangle((x, y), studs * 0.55, 0.5, facecolor=colour,
                                   edgecolor=EDGE, linewidth=1.0, zorder=2))
        for s in range(studs):
            ax.add_patch(Circle((x + 0.275 + s * 0.55, y + 0.5), 0.13,
                                facecolor=colour, edgecolor=EDGE,
                                linewidth=0.8, zorder=3))

    # each step adds exactly one brick to what the previous step produced
    plan = [[("#dc2626", 4, 0.0)],
            [("#dc2626", 4, 0.0), ("#f59e0b", 3, 0.62)],
            [("#dc2626", 4, 0.0), ("#f59e0b", 3, 0.62), ("#2563eb", 2, 1.24)]]
    ax.text(4.0, 4.5, "Assembly instructions", ha="center", fontsize=10,
            color=MUTED)
    for step, bricks in enumerate(plan, start=1):
        x0 = 0.4 + (step - 1) * 2.9
        ax.text(x0 + 1.1, 3.85, "%d" % step, ha="center", fontsize=11,
                color=INK, fontweight="bold")
        for colour, n, dy in bricks:
            brick(x0 + (4 - n) * 0.275, 1.6 + dy, n, colour)
        if step < len(plan):
            _arrow(ax, [(x0 + 2.35, 2.2), (x0 + 2.75, 2.2)])
    ax.text(4.0, 0.9, "Each numbered step is one unambiguous action, carried out\n"
                      "in order — an algorithm written entirely in pictures.",
            ha="center", va="top", fontsize=9, color=INK)
    _frame(ax, (-0.2, 8.4), (0.0, 4.9))

def draw_safety_card(ax):
    """Stands in for aircraftsafety.jpg — a photo of an airline safety card.

    Stylized, not the original photograph (see FIGURES.md).
    """
    DARK = "#0f172a"
    panels = ["fasten\nseatbelt", "locate\nexits", "pull\nmask down",
              "breathe\nnormally"]
    for k, label in enumerate(panels):
        x = k * 2.4
        ax.add_patch(plt.Rectangle((x, 1.4), 2.1, 2.4, facecolor="#fef9c3",
                                   edgecolor=EDGE, linewidth=1.2, zorder=1))
        # a seated figure, common to every panel
        ax.add_patch(Circle((x + 1.05, 2.95), 0.3, facecolor=DARK,
                            edgecolor="none", zorder=2))
        ax.add_patch(plt.Rectangle((x + 0.75, 1.95), 0.6, 0.75, facecolor=DARK,
                                   edgecolor="none", zorder=2))
        # ...then one distinguishing pictogram per step
        if k == 0:                                   # belt across the lap
            ax.add_patch(plt.Rectangle((x + 0.6, 2.0), 0.9, 0.16,
                                       facecolor="#dc2626", edgecolor="none",
                                       zorder=3))
        elif k == 1:                                 # arrow to an exit
            _arrow(ax, [(x + 1.4, 2.35), (x + 1.85, 2.35)], color="#dc2626")
            ax.add_patch(plt.Rectangle((x + 1.85, 1.95), 0.14, 0.8,
                                       facecolor="#16a34a", edgecolor="none",
                                       zorder=3))
        elif k == 2:                                 # mask dropping down
            ax.add_patch(Circle((x + 1.05, 3.55), 0.17, facecolor="#dc2626",
                                edgecolor="none", zorder=3))
            _arrow(ax, [(x + 1.05, 3.38), (x + 1.05, 3.12)], color="#dc2626")
        else:                                        # mask on, breathing
            ax.add_patch(plt.Rectangle((x + 0.88, 2.82), 0.34, 0.24,
                                       facecolor="#dc2626", edgecolor="none",
                                       zorder=3))
        ax.text(x + 1.05, 1.6, "%d" % (k + 1), ha="center", fontsize=9,
                color=INK, fontweight="bold")
        ax.text(x + 1.05, 1.25, label, ha="center", va="top", fontsize=8,
                color=INK)
    for k in range(len(panels) - 1):
        _arrow(ax, [(k * 2.4 + 2.13, 2.6), (k * 2.4 + 2.37, 2.6)])
    ax.text(4.35, 4.35, "Safety card: an algorithm with no words at all",
            ha="center", fontsize=9, color=MUTED)
    _frame(ax, (-0.3, 9.6), (0.4, 4.7))

def draw_notebook_repl(ax):
    """Replaces the PyCharm interactive-mode screenshot (02-interactivemode.png).

    Redrawn as the Jupyter/Colab notebook the course actually uses.
    """
    _mockwindow(ax, 10.5, 5.4, "CS1910_Chapter.ipynb  —  Jupyter / Colab", [
        ("In [1]:  print(\"Hello, World!\")", ACCENT),
        ("         Hello, World!", INK),
        ("", None),
        ("In [2]:  2 + 2", ACCENT),
        ("Out[2]:  4", WARM),
        ("", None),
        ("In [3]:  name = \"Alice\"", ACCENT),
        ("         print(\"Hello,\", name)", ACCENT),
        ("         Hello, Alice", INK),
    ])
    _frame(ax, (-0.3, 10.8), (-0.3, 5.7))

def draw_script_run(ax):
    """Replaces the PyCharm script-mode screenshot (02-scriptmode.png)."""
    _mockwindow(ax, 5.6, 4.4, "hello.py", [
        ("name = input(\"Your name? \")", ACCENT),
        ("greeting = \"Hello, \" + name", ACCENT),
        ("print(greeting)", ACCENT),
    ])
    ax.text(2.8, 0.45, "a script saved to a file", ha="center", fontsize=8,
            color=MUTED)
    _arrow(ax, [(5.9, 2.2), (7.5, 2.2)], "run it", label_off=(0, 0.3))
    ax2 = ax
    ax2.add_patch(plt.Rectangle((7.8, 0), 5.6, 4.4, facecolor="#111827",
                                edgecolor=EDGE, linewidth=1.2, zorder=1))
    ax2.add_patch(plt.Rectangle((7.8, 3.85), 5.6, 0.55, facecolor="#374151",
                                edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax2.text(8.0, 4.12, "Terminal", fontsize=9, va="center", color="white",
             zorder=3)
    for k, (line, colour) in enumerate([
            ("$ python hello.py", "#93c5fd"),
            ("Your name? Alice", "#e5e7eb"),
            ("Hello, Alice", "#86efac")]):
        ax2.text(8.05, 3.35 - k * 0.5, line, fontsize=9, va="center",
                 family="monospace", color=colour, zorder=3)
    _frame(ax, (-0.3, 13.7), (-0.3, 4.7))

# ---------------------------------------------------------------- runtime ---
_SIZES = {'lego_instructions': (6.6, 4.4), 'safety_card': (7.4, 3.8), 'notebook_repl': (7.4, 4.0), 'script_run': (9.4, 3.4)}
_WIDTHS = {'lego_instructions': 620, 'safety_card': 660, 'notebook_repl': 700, 'script_run': 760}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in ['lego_instructions', 'safety_card', 'notebook_repl', 'script_run']:
    _FIGURES[_n] = _render(_n)

print("Setup complete \u2014 4 figure(s) ready.")


### Introduction to Computational Thinking

Welcome to Computer Science 1910 at the University of Prince Edward Island.  Our team welcomes you to what will be an eye-opening experience in how to think about, analyze, and solve problems.

Wait... what did you just read?  Isn’t this class about programming?  Wasn’t this the class where you were going to learn the arcane instructions that make all of our wonderful toys, from game consoles to self-driving cars, work?  Isn’t this the scary class that is super hard? Will this class unlock the secrets of the wizards of Silicon Valley and enable you to make your own fortune?  Well, the answer to that, like everything else is: it's complicated!

Coming into this class, you may be in one of our Computer Science programs, having come to the university expecting that most of what you will learn is programming so you can build some of those previously mentioned toys.  Alternatively, you might be taking this class because it is part of the common core of one of our Mathematics, Statistics, Analytics, or Actuarial Science programs, and you have been told you have to take this class as part of your program.  Finally, it might be that you have come to this class with simply an interest in taking an option that will stretch your knowledge beyond your degree program.  Whatever your reasons for joining us, you will quickly find that computer science is much more than just learning to program.  You are going to learn a set of life skills that will help you solve problems regardless of if you have a future as a programmer or not.

#### Computer Science and Computation

What is the science of computers?  When we use the word "Science’’ we often think about the natural sciences such as biology, chemistry and physics.  As young people, we are told the story about how Newton observed an apple falling out of a tree and came up with the notion of gravity, and then rockets taking us to the moon happened shortly thereafter.  That story highlights one of the keys of the natural sciences: they are trying to understand the why and how of the observations that we make in the world around us.

Well, what then is *computer science*?  Are we studying the why and how of the boxes on our desks?

Computer science comes from a different pedigree from the natural sciences.  Computer science is a fusion of mathematics and engineering where we study how to do *computation*, which can be defined as any process where you undertake a set of steps, some arithmetical, some non-arithmetical, to calculate some kind of outcome.

While it is tempting to think of the boxes on our desks, in our cars, or in our fridges as being what computation is, in fact, humans have been doing computation for as long as we have needed to do something that takes more than 10 fingers.  We have records of computation being done as early as 37,000 years ago throughout Africa with tally sticks carved in a bone as a means of tracking numbers.  In Eastern Asia, we find examples of counting sticks that represented rod numerals, one of the earliest true positional counting systems that allowed tracking of very small or very large numbers. As civilizations progressed, we saw the introduction of computation for doing astronomical calculations in the Hellenistic, Roman, Egyptian and other cultural histories, with one of the most complex being the castle clock by Ismail al-Jazari, the first analog programmable computer, predating the Babbage Engine, one of the earliest models of computing in Europe, by nearly 600 years.  In each case, the "computer’’ was created to help solve a problem, with a set of rules and steps that needed to be followed in order to reach an answer --- whether it was how many yields of grain came in the last season, or when the moon was going to reach its pinnacle.

You might be (rightfully) wondering: how does this connect to Newton’s apple?  We can conceptualize this study of computation in the following way: whereas natural sciences try to describe the observations in our universe, computer science invents its universe, however large or small, and then tries to describe the how and why of what has been created.  In mathematics, we are often creating models that describe and predict how computation will behave under different rules and constraints.  On the other hand, most computation is now done digitally, using electronics engineered to represent and process computational steps.  In an oversimplified description, computer science in these cases are often studying how to build better versions of those electronic systems or trying to understand how to solve problems using them.

### Computational thinking

When we are looking to solve a particularly hard problem, we can use computation to help us solve it.  However, before we can use computation, we first need to translate the problem into a form where computation can be done, in the same way that Al-Jazari needed to translate observations of the sun, moon and stars into floats, pulleys, levers and valves in order for the clock to track the day and night sky (among other things).

This seemingly difficult task is a process known as *computational thinking*, where we systematically analyze and eventually automate problems for computation.  While the principles and the term itself have been bandied about in technology since the 1950s, it is only in the last 15 years that it has been conceptualized as a key skill for people outside of computer science to develop and use.  In some countries, such as England, computational thinking has been part of their national curriculum since 2011, however, in the majority of countries, it is something that is left for computer science students alone.  In the following sections, we will discuss some of the aspects of computational thinking so you can apply them in solving problems.

#### Concepts of Computational Thinking

There are a variety of problem-solving skills that are used when working with any problem.  We could start with the Castle Clock, but that seems a little bit difficult.  Let’s instead start with a problem that many people have encountered: learning a new card game, such as *Go Fish!*  This is a game where one player chooses a card in their hand and asks an opponent for a card of the same rank.  If the opponent has that card, they must pass the card to the asking player.  If they do not, the asking player needs to draw a new card from the deck.  When a player has 4 of a particular rank, then they lay those cards on the table as a "book".  Each round every player gets 1 turn until the person who lays down all their cards first wins.

At first glance, this seems like a relatively simple game.  However, think about how you might teach someone else *Go Fish!* who has never encountered any type of card game before.  First, you would need to break down the game into smaller components.  You will need to explain what a deck is, what a hand is, what a book is, and even what a card is with a suit and a rank.  You would need to *decompose*   the game into its individual parts.

However, just knowing those components does not help someone play the game of *Go Fish!*  We have to explain the rules, such as how you ask for cards, when cards can be passed between players’ hands, and when a player can lay a book of cards down on the table.  These *patterns*  of play allows you to play the game repeatedly one player after another using the same rules.

Now, in addition to these rules, you will need to teach the new player some of the finer nuances of *Go Fish!*  The new player will need to listen to what other players are asking for, and use *logic*  in conjunction with the rules to make predictions about what is going to happen on subsequent turns.  This is combined with the player *evaluating*  the outcomes of each turn to make decisions about whether a win is a likely outcome.

Finally, combining all of the above, components, the patterns, the logic and the evaluations are *abstracted*  into a series of turns that comprise the game.  The game itself can be described as an algorithm comprising of a sequence of turns that repeat until there is a winner.

```

Algorithm PlayGoFish!:

deal 5 cards to each player
choose a current player
as long as no player has an empty hand: 
    current player takes turn
	if the current player has cards in their hand:
		make the other player the current player		 
the current player wins


```

Now the really interesting thing is, when a player learns one card game, they can re-use a lot of the reasoning they have picked up to learn other games, further abstracting up a level into types of card games.  Trick-taking card games are about getting the most books of cards, solitaire games are about emptying the deck and sorting the cards by suit or by rank, poker games are about having the highest hand, and so on.

> **Concepts of Computational Thinking**
>
> What has just been laid out is an example of the application of computational thinking, showing the key concepts of computational thinking:
>
> **Algorithm**: A sequence of instructions on how to complete a task (e.g. playing a game to a winner).
> **Abstraction**: Identifying the important details that are needed to reason about a problem and hiding the underlying detail (e.g. a round is a sequence of turns where every player gets a turn).
> **Decomposition**: Breaking a problem down into smaller pieces to make a task tractable to solve in smaller parts (e.g. cards, deck, hand, book).
> **Patterns**: Noticing sequences of actions that recur in order to use them to make predictions as to what will happen next (e.g. rules of a player’s turn).
> **Logic**: Establishing and checking facts in order to inform predictions (e.g. identifying cards in other’s hands).
> **Automation**: Translating specifications of solutions to problems into something that can be executed by a machine. Most commonly this is programming code.
> **Evaluation**: Using the facts and patterns to evaluate the future outcomes (e.g. how close is the game to ending).

Each of these skills is something that you likely apply in your everyday lives.  For example, if you were asked to set a table for dinner, you would probably start thinking about the items you would need to put on a table.  Depending on where the dinner is taking place, who is attending, the culture of the food, and any number of other variables the actual items that end up on the table will change, but you will abstract, decompose, follow rules and patterns and evaluate (or have someone else evaluate) whether the setting is correct.

When we approach a complex problem we need to apply these skills to understand it.  Often, after analyzing those problems, we find they are either too time-consuming or complex for us to finish on our own, at which point we turn to automation to solve them.  At that point, we need to translate our understanding into something the digital computer can understand.  For that, we need a different set of skills.

#### Behaviours of Computational Thinking

When we translate a problem into a computer we are doing something creative.  One of the truly wonderful things about digital computing is that we are at a point where we can create remarkable things, and almost anything our imaginations can conjure.

However, the road from the first line of code to an incredible magical system is a bumpy one from time to time.  You will spend a lot of time *tinkering*  with code, changing things in strategic ways to see what happens.  This will be a process of experimentation, where some things will work, and some will not, but you will learn something new when you do, either about your code or about programming in general.

Other times, you will be trying to fix a problem in a systematic way in a process that is called *debugging* .  Sometimes it will be simple, and you will figure it out on the first try.  At other times, you will spend quite a long time trying to find and fix the bug.  When that happens, you will learn the most important skill - that of *perseverance* .

#### Linking Computational Thinking and Programming

Now, with all of that said, you may be thinking to yourself: "If this is all about thinking and solving problems, why do I have to learn to program at all?’’  This is a valid question, and we want to take a few minutes to explore the reasons why we teach programming.

The first, and perhaps most important reason, is that programming is fertile ground in which you can try many of the thinking skills and behaviours we talked about in the previous sections, and you can do it almost without consequence.  You can create things, tinker with them, debug the problems and persevere when it gets hard, and know that you basically can’t break anything.

The second reason that we teach programming is that it helps you understand how the technology around you works.  While you may not be able to program the interface for a smart fridge, by the end of the course you will have at least some inkling of what the smart fridge is doing when it cycles your weekend selfies across the door.  Having that depth of knowledge technology is important because, in a society where we depend on technology every day, the populace needs to know what that technology is doing.  Without this understanding, technology is indistinguishable from magic, and a source of fear and superstition, as we have seen in many movies where villains use a sinister "algorithm" to thwart the heroes.

The final reason we teach programming for our whole first-year cohort is that it is a useful skill for a wide variety of applied fields in which you may end up working.  Clearly some of you are targeting careers in technology such as software engineers and game developers.  However, beyond these obvious destinations for programming skills, there are many industries where a basic knowledge of programming will give you a leg up on the competition.  Whether you are working as a statistician informing government policy from an organization like Institute for Clinical Evaluation Services (ICES) or as a Risk Analyst for Canadian Tire, programming can help you automate both mundane tasks and complex reasoning.

#### Onward

Here at UPEI, we have the unique opportunity to provide an entire community of students these key skills and behaviour in computational thinking.  Even if you never write a single line of code after your first year, the terminology, the problem-solving strategies, and the skills you gain here will influence how you solve problems in your respective fields for years to come.

We start this journey in the next chapter with an in-depth look at one of the key computational thinking concepts: algorithms.

### Algorithms

It is usually the case when we are solving problems that we are not going to solve it just a single time.  We often are trying to solve the same problem repeatedly or we are trying to solve a problem which is similar to one we have encountered before.

We see examples all over the world of procedures and processes that have been put in place to help people solve problems.  For example, a company like IKEA sells furniture that comes with a preset number of parts and a set of language independent instructions that allow people to assemble furniture in their own home.  Presumably, before this happened, someone needed to work out what furniture that was needed (e.g. a desk, a planter, a cat house), what parts were needed, the order in which assembly needs to happen, and then produce the manual that describes that order so that the construction can be repeated.  At IKEA, they have gone one step further.  They try to create things within their solutions that are reusable, everything from brackets to sections of the manuals that can be re-purposed for the next product.  Throughout this process, the IKEA designers and engineers undertake computational thinking to construct a new products.  In order to ensure that everyone can construct exactly the same desk, planter or cat house, IKEA relies on sequences of steps that are *algorithms*.

An *algorithm* is a list of actions that describe how to perform a task or solve a problem.  Turning to a different domain, that of preparing food to eat, we find great examples of algorithms.  For instance, a recipe for making bread is an algorithm.   The recipe describes what actions you must take, and the order in which you must take them, if you want to end up with something that looks and tastes like bread.  If you deviate from the algorithm, there's a good chance you end up with something quite un-bread-like.  Other examples of algorithms are:

- the rules of a board game;
- steps to operate a coffee maker; and
- a list of things to do in case of a fire.

Here is a concrete example showing the specific steps in an algorithm to make ramen noodles:

```

Algorithm MakeRamen:

boil water
add noodles to water
wait 6-8 minutes
drain the noodles
stir in contents of flavour packet
place cooked noodles in bowl

```

This algorithm consists of six actions to solve the problem of making ramen noodles.  The actions are taken in the order given, and the end result, or *output* of the algorithm is a prepared bowl of steaming hot noodles, ready to eat.  The important thing to remember about algorithms is that the given actions must be taken in the given order, otherwise you are not following the algorithm and likely will not get the desired output.

An algorithm must satisfy the following three qualities:

- Executable : It must be the case that a machine (which could include a human machine) can follow and complete each step of the algorithm.
- Unambiguous : When the machine reaches an instruction in an algorithm, there must be no question about what must be done to execute the instruction.
- Terminates : An algorithm must have a point at which it stops.  Otherwise, the algorithm will run forever and will never yield the desired outcome.

#### Blocks, Repetition, and Conditionals

Algorithms can contain things other than just actions to take, such as setting conditions under which actions should be taken.  The use of logic in these conditions let us know when particular actions can and cannot happen.  Let's consider the following algorithm for playing the game Jenga.  Jenga is a two-player game where you start with a tower of blocks, and players take turns making the tower taller by removing an existing block from the tower and placing it on the top of the tower without knocking it over.  The player who knocks down the tower loses.  For more on how to play Jenga, [you can watch an insightful video explanation](https://www.youtube.com/watch?v=I7H6wGy5zf4).  Here's an algorithm for playing Jenga:

```

Algorithm PlayJenga:

stack the blocks to form a tower
choose a current player
as long as the tower of blocks is still standing: 
    current player takes turn
	if the tower is still standing:
		make the other player the current player		 
the current player loses

```

In this algorithm we have a sequence of actions that needs to be repeated as long the tower is still standing. At the end of the repeated sequence players must evaluate whether or not the game is over (i.e. the tower has fallen) or if they need to keep playing.

Notice how the actions to be repeated are indented.  Such sequence of actions is called a *block* . So in this algorithm we have a block of two actions to be repeated. Notice that the second action in the block, the one that begins with `if the tower...`, is not actually an action, it is a condition that says when some other action, `make the other player the current player`, should be performed.  Notice how this action is indented again, relative to the actions in the block of repeated actions.  This means that the action `make the other player the current player` is, itself, a single-action block where the action is only taken if the condition `the tower of blocks is still standing` is true.  A block inside of another block is called a *nested block*.  Blocks can be *nested*  to any number of levels. Finally, we can see that the action `the current player loses` is not one of the actions to be repeated because it is not indented. Indentation is almost always used to denote blocks. (There are other ways to denote blocks in algorithms besides indentation such as enclosing a block in a pair of curly (accolade) braces  { ... \).  But even then, indentation is almost always used as well to make algorithms more readable to humans.}  Thus, when reading algorithms, be aware that the indentation is not arbitrary or accidental, but rather conveys important meaning.

Now, consider that that algorithm doesn't actually describe in much detail what the player actually does.  The information about a player's turn is just like the turn in *Go Fish!*.  It abstracts away the details of what happens, and thus it is hard to know exactly what is happening.  However, when we *decompose* the player's turn, we get a very different of the steps of a player's turn:

---

```

Algorithm PlayJenga:

stack the blocks to form a tower
choose a current player
as long as the tower of blocks is still standing: 
    current player takes a block from middle
    current player puts the block on top
	if the tower is still standing:
		make the other player the current player		 
the current player loses

```

A different level of abstraction can often serve a very different purpose.  This more detailed algorithm can now act as an instruction book for teaching someone the game of Jenga, while the former might be one that we would use if we casually describing it to someone at a board game themed party where people know a lot about how board games work.

#### Variables

Algorithms often deal with numbers and are easier to write if we are allowed to give names to numbers.  A *variable*  is a name given to a particular value such as a number.   In the next chapter we will talk about how variables can be many different types of values, but for the moment we will stick with numbers.

We can write algorithms with variables that refer to values, change the values they refer to whenever we want, and use them them to compute other values.  For example, here's an algorithm for computing the average of a set of numbers that uses a variable to keep track of a running sum:

```

Algorithm Average:
Input: a set of numbers 

let total = 0;    
for each number x in the set of numbers:
	let total = total + x
let average = total / size of the set of numbers

```

In this algorithm the first action, `let total = 0`, means that we associate the variable name `total` with the value 0.  Then we have a single-action block which is repeated for each number `x` in the input set.  That action adds `x` to the existing value referred to by `total` and associates `total` with the new resulting value.  Then the *output* of the algorithm, the average, is associated with another variable called `average`.

#### Input and Output

In the algorithm in the previous section, note how we explicitly specified the input to the algorithm in the second line.  In general, algorithms are allowed to have any number of *inputs* (including none).  Inputs are data that **come from outside the algorithm** which the algorithm uses to complete its task.  An algorithm with inputs is often much more useful than an algorithm that has none.  For example, here is an algorithm that computes the average of the set of even numbers between 1 and 10:

```

Algorithm AverageEven1to10:

let total = 0
for each number x in the set {2, 4, 6, 8, 10}:
	let total = total + x
average = total / 5

```

This is exactly the same algorithm as the `Average` algorithm in the previous section except that it does not have any inputs and it only works for one specific set of numbers which is written right into the algorithm.   By allowing the set of numbers to be an *input*   to the algorithm, we get an algorithm that is reusable in vastly more situations and can compute the average of any set of numbers instead of just one specific set.

Algorithms have one or more *outputs* .  An *output* is data that is the result of the task that the algorithm was intended to carry out.   In the `Average` algorithm from the previous section we didn't explicitly specify that the variable `average` is the algorithm's output, but we could, for example:

```

Algorithm Average:
Input: a set of numbers 
Output: a variable 'average' which refers to the average
        of the numbers in the input set

let total = 0;    
for each number x in the set of numbers:
	let total = total + x
let average = total / size of the set of numbers

```

Now the output of the algorithm is explicitly specified.  If an algorithm has more than one input or output, then the additional inputs and outputs can be described in the same fashion.

Notice that for every algorithm that we have shown you so far, we have given it a name, and sometimes specified its input and output, and these things have appeared before the algorithm's first action.  The description of an algorithm's name, inputs, and outputs, are collectively called the algorithm's *header*.  The items in the header are not actions in the algorithm to be carried out, but rather describe the algorithm and its intended usage.

#### Methods of Writing Algorithms

Algorithms can be written in different forms.  So far we've seen a few algorithms that are written in words.  Algorithms written in words are called *pseudocode* .  Pseudocode can look like the "code" we would write in a programming language, but is much more flexible because its syntax and form is not rigidly specified like that of a programming language.

Pseudocode is often used as a rapid prototyping tool for an algorithm. By rapid prototyping, we mean that you can get to a solution, even if it is far from perfect, very quickly.  We can then work with different levels of abstraction, or different types of refinements, change the variables and so on with nothing more than a piece of paper, a pencil and an eraser.

The LEGO instructions and the airline safety card pictured in Figure  are examples of algorithms written using pictures.  They indicate, in a step-by-step manner, what to do to complete the tasks of building the LEGO model, and escaping the aircraft in an emergency.

In [ ]:
show("lego_instructions")
show("safety_card")

*Figure: Examples of algorithms written using pictures.  Left: LEGO instructions; right: aircraft safety procedures card.*

Many types of problems require *sequential solutions* (The alternative is to solve a problem with a *parallel solution), where multiple steps happen at once. These are common in large problems involving large amounts of processing or data --- these are covered in later courses.*, meaning that one step happens after another in order without deviation, which is a common feature in a lot of different fields.  Due to this, you have probably already encountered one of the most common forms of pictures based algorithms: a flowchart.   Flowcharts working at different levels of abstraction can help us describe how and when different events can occur.  You will see throughout these readings there is a flowchart language that is developed throughout to help you describe what is happening on your programs.  Prior to ever writing a line of code, working through a flowchart can often help you work out where those key logic problems are likely to be.

Words and pictures are normally how we write algorithms that are to be understood and/or carried out by humans.  When we write algorithms for computers we have to use a language that a computer can understand.  Computer programs are written in a *programming language*.  A programming language provides instructions to the computer in a way that it can both understand them and carry them out unambiguously.  For this reason, computer programs are much more strict in the syntax and style we can use to write algorithms.

If a computer program doesn't do what its programmer wants it to, it's not the computer's fault!  The computer can only do exactly what the program tells it to do whether or not is what the programmer intended.  This is why it is vital that a programmer understand the algorithm that he/she is trying to write.  If a programmer doesn't understand the algorithm, the chance that they'll be able to tell the computer how to perform the algorithm correctly is slim to none.  This is why it is advantageous for programmers to write algorithms using pseudocode first .  It helps them to ensure they understand what they are about to program without having to worry about the details of the programming language syntax.  Once understanding is reached via pseudocode, a programmer is much more easily able to get the details right when writing the algorithm in the precise syntax required by the programming language.  Moreover, they are ready to implement the algorithm in **any** programming language that they know!

In this course we will be using the Python programming language.  Let's look at what the `Average` algorithm looks like when it is translated from pseudocode to Python. Don't worry if you don't understand **why** the algorithm is written the way it is in Python.  We'll get to that later.

```

Algorithm Average:
Input: a set of numbers 

let total = 0;    
for each number x in the set of numbers:
	let total = total + x
let average = total / size of the set of numbers

```

In [ ]:
def Average(S):
	total = 0
	for x in S:
		total = total + x
	average = total / len(S)
	return average

You should be able to appreciate that the two versions of the algorithm are doing the same thing.  The header in the pseudocode algorithm has been translated to the line starting with `def` in Python, so if you're thinking that this is some Python syntax for saying that we want to define an algorithm, give it a name, and say what its inputs are, then you're right.  The `S` in the round brackets indicates that {S} is the input to the algorithm.  You should also be able to appreciate that `total` and `average` are variables in the Python version, just as they are in the pseudocode version.  The repetition `for x in S` looks much the same as in the pseudocode and indicates that we do the same thing to each element of the set `S`.  The `len(S)` syntax tells us how many numbers are in the set `S`.  The blue words in the Python code have specific, well-defined meanings in Python.  The last line contains the command `return` which tells Python that the variable `average` is the output of the algorithm.

The Python code can be understood and carried out by a computer, but the pseudocode algorithm cannot, even though it doesn't look that different.  Depending on how a pseudocode algorithm is written, there may or may not be an easy, line-by-line translation of the pseudocode algorithm into Python (though in this case, it's pretty close).

### Abstraction and Refinement

We saw earlier how a single algorithm can be written at several different levels detail or, as computer scientists refer to this, levels of *abstraction*of abstraction.   Abstraction is the process of hiding of details that are not currently important.  To illustrate this concept, let's look again at our `MakeRamen` algorithm.

---

```

Algorithm MakeRamen:

boil water
add noodles to water
wait 6-8 minutes
drain the noodles
stir in contents of flavour packet
place cooked noodles in bowl

```

The first instruction in this pseudocode is `boil water`.  This is a great example of abstraction, because the action `boil water` glosses over all of the details of **how** to boil water.  For humans, these details are pretty unimportant, because adult humans all know how to boil water.  But imagine that this algorithm has to be carried out by a humanoid cooking robot.   The robot does not intuitively know how to boil water.  The action `boil water` is too *abstract* for it.  It needs more details.

The process of describing more detail about how an instruction should be carried out is called *refinement*.  For example, we might refine the `boil water` action by replacing it with a sequence of actions (shown in red text) that describe how to boil water in more detail:

```

Algorithm MakeRamen:

place pot under faucet
add water to pot
place pot on stove
turn on burner
wait until water boils
add noodles to water
wait 6-8 minutes
drain the noodles
stir in contents of flavour packet
place cooked noodles in bowl

```

Each of these new actions describes an action that partially carries out the original `boil water` action.  But even these actions may not be detailed enough for our robot to carry out the task.  At some point, the robot needs to know exactly where, and for how long to position its legs and arms to carry out these actions.   This would require that we further refine the actions `place pot under faucet`, `add water to pot`, etc. to the level of detail where we tell the robot exactly where and how to move its limbs by replacing each of these actions with sequences of even more detailed actions.  This is called *stepwise refinement* .  We repeatedly replace actions that are too abstract with a decomposed sequence with more detail until we reach a level of detail that can be directly carried out.   Each level of refinement results in actions that are at a lower level of abstraction and are closer to the individual actions that the robot (or computer) can carry out natively.

The ability to think at different levels of abstraction and mentally move between them is critical to success to solving problems.  We abstract away details when they are not important, and refine abstractions later when we are ready for the detail.

Defining an algorithm and giving it a name is, itself, a form of abstraction. Naming an algorithm and describing its inputs and outputs allows someone to use the algorithm without knowing **how** the algorithm works.  In other words, an algorithm, once written, hides the details of how the algorithm is performed, allowing it to be used to produce results without knowledge of the algorithm's details.  For example, having defined the algorithm `Average`, we no longer have to remember that to compute an average, you add all the numbers up and divide by how many numbers there were.  All we have to do is say "perform the algorithm `Average` on the set of numbers 1, 2, 3, 4, and 5", and we'll get the correct answer of 3.

We take abstraction for granted all the time.  So many of the things we do on computers that look really simple are actually abstractions of breathtakingly complex algorithms and hardware details.  These are things like Google Search, fingerprint ID on your phone, and face recognition in your digital photography software.  There is a tendency for people who are used to such technology to underestimate its complexity.  Keep this in mind the next time you think to yourself that it would be really "easy" to add some desired feature to your phone!

To summarize, abstraction allows us to think about performing higher-level, more complex actions without worrying about **how** they are performed.  Abstraction  doesn't mean that the lower-level details of how an abstracted algorithm is carried out don't exist or never have to be written at some point.  It is just a mechanism that allows us to ignore such details when it is convenient or until they are needed.

### Problems to Algorithms

It is important to distinguish between problems and algorithms.  A *problem* is a task to be carried out.  An *algorithm* is a specific set of steps for **how** to carry out a task.   A problem may have more than one algorithm for solving it.    A given algorithm, however, solves only one problem.

Let's consider the problem of picking up a pile of playing cards that you were just dealt, and putting them in rank-order.  Some people pick up their cards one at a time and place each card into their hand at the correct position as they do so.  Other people pick up all of the cards at once, find the smallest one and move that card to the left-most position, then find the next smallest and put it next to the smallest card, and so on.  These are two different algorithms for solving the problem of putting a hand of cards in order.  Both achieve the same result, but the algorithms themselves are fundamentally different processes.

### Algorithms to Programming

Once we have chosen an algorithm, such as the example of constructing a hand of cards, we can transfer that algorithm into something that can be executed by a computer.

This is where we take our first steps towards programming.  Programming is our means of expressing the solution to a problem, which is expressed as an algorithm, in a way that is *clear*, *accurate*, and *automates* the solution.  Implementing an algorithm in a program allows us to reuse our solutions again and again, and the creative act of programming is one way that we can work through the finer points of a solution.

For example, let's say that we had a bunch of bags of marbles and we wanted to sort the bags from heaviest (the most marbles) to the lightest (the least marbles), but we can't look in the bags.  We pick up 2 bags, shake them, and put the one that sounds like it has more marbles first, then the other one second.  Then we compare a third bag to each of those, and put it in the right place.  We can encode this in a programming pretty easily.  However, computers require very precise instructions in order to work, so while creating a program to solve this simple problem we will be forced to ask questions like:

- "What happens if bag has only 1 marble and doesn't make a sound?"
- "What happens if a bag has no marbles? Does it count as a bag of marbles?"
- "Does each bag need to be compared to all of the bags previously put in order?"

On one level, you will be learning to program, a useful skill by itself. On
another level, you will use creative act that is programming as a means of reasoning and solving problems.

#### The Python programming language

The programming language you will be learning is Python. Python is an example
of a *high-level language*.  The most recent version is Python 3, and represents an evolution of languages that started back in the 1970s.

You may have heard stories about how the precursor to programming was originally done through punchcards.  Programmers would carefully punch out a card representing a character and put it in a deck with other cards to build up a program.  This was clearly time consuming, and prone to mechanical and a variety of programmer errors such as placing cards out of order.  Despite this, programming on cards would persist until the mid-1970s.

As electronic computing came to the fore, a set of *low-level languages*  emerged.  These machine languages, or *assembly languages* often allowed programmers to encode things at the level of the chip through a series of unique codes.  These programming languages were in common use well into the 1990s, and still some are used today in many legacy systems (A *legacy system)  is a system that was written at some point in the past where it had a very specific purpose, and is maintained for those who still need to do that task.  Often, there are newer solutions for legacy systems, but change in various industries is very slow, and so maintenance of systems from the 1970s persists today in some industries*.  and were the primarily way of programming early game consoles like the Atari 2600.  However, while these languages allowed very tight levels of control by the programmer, they were inefficient from the point of view of creating programs.  Common things that programmers needed, like the ability to repeat sections of code or make decisions in code, had to be individually managed by the programmer.

In the 1950s the invention of *high-level*  programming languages abstracted away these low level codes.  Whereas previously a programmer would have to track where to jump in their code for repeating a section (e.g. jump to line 85 ---`GOTO 85`), programmers could write code in a way that increasingly was like a human readable written language, making code faster and easier to write and maintain, and also code that was more portable.  Code written in a high level language can be translated into representations that can be read by many different types of computers provided there is an engine to translate that language to meet the architecture of the computer.

There are many such languages, with many of the common ones you have probably heard of are C, C++, Pascal, PHP, Java and of course ... Python.  There are several nuanced differences between these languages and how they work.  We call some of these out for people who are already familiar with programming throughout the book, however largely these differences are taught later in the computer science program.

The key commonality of all of these languages is that they need to be translated into a lower level representation for the machine to understand.  In the case of Python, we refer to the engine that does this translation from *source code* to *bytecode* and then executes it as an *interpreter* .

> **Compilers versus Interpreters**
>
> For those familiar with programming languages, you may have used a language like C, which is a *compiled* programming language.  In a compiled language, the programming code is usually translated to a very low-level machine code representation that is tied very closely to the computer processor.  As a result, a program written in a programming language like C that is compiled on a Windows machine will not run on an Apple.
>
> In comparison, an *interpreted* programming language is translated first into what is known as *bytecode* .  Bytecode can be moved from machine to machine, and as long as there is an interpreter to read and execute the bytecode, you do not need the original source code program.

There are two ways to use the Python interpreter: *interactive mode* (sometimes called *immediate mode*) and *script mode*. In immediate mode, you type Python expressions into the Python Interpreter window,and the interpreter immediately shows the result:

In [ ]:
show("notebook_repl")

*Figure: a notebook in interactive use — `2 + 2` and the result `4`.*

The `{>}{>}>` is called the *Python prompt*. The interpreter uses the prompt to indicate that it is ready for instructions. We typed `2 + 2`, and the interpreter evaluated our expression, and replied `4`,
and on the next line it gave a new prompt, indicating that it is ready for more input.

Alternatively, you can write a program in a file and use the interpreter to
execute the contents of the file. Such a file is called a *script* .   Scripts have the advantage that they can be saved to disk, printed, and so on.

In the UPEI introductory Python courses we work in Jupyter notebooks, either in Google Colab or in JupyterLab.  Notebooks let you mix runnable code, its output, and explanation in one document — which is exactly what this book is.

For example, we created a file named *firstprogram.py* using the notebook.
By convention, files that contain Python programs have names that end with
`.py`.

When you execute the program  (Either by pressing the green triangle at the first line of the script or through a hot key (Control-R on Windows, Command-R on Apple).), the program prints its initial line and then the answer to 2+2 as it did before in the console.

To execute the program, we can click the "Run" button, a green triangle placed at the first line of the script in the notebook:

In [ ]:
show("script_run")

*Figure: a script saved as `hello.py`, and the terminal that runs it.*

Most programs are more interesting than this one.

Working directly in the interpreter in interactive mode is convenient for testing short bits of code because you get immediate feedback. Think of it as scratch paper used to help you work out problems. Anything longer than a few lines should be put into a script.

#### What is a program?

A *program*  is a sequence of instructions that specifies how to perform a
computation. The computation might be something mathematical, such as solving a
system of equations or finding the roots of a polynomial, but it can also be a
something more complex such as searching and replacing text in a document.

The details look different in different languages, but most high level languages have a means of doing each of the following:

- **Input:**  Get data from the keyboard, a file, or some other device.
- **Output:**  Display data on the screen or send data to a file or other device.
- **Math:**  Perform basic mathematical operations like addition and multiplication.
- **Conditional execution:**  Sometimes referred to as "decision" points in code or branching, the program checks for certain conditions and execute the appropriate sequence of statements.
- **Repetition:**  Perform some action repeatedly, usually with some variation.

Believe it or not, that's pretty much all there is to it. Every program you've
ever used, no matter how complicated, is made up of instructions that look more
or less like these.   As discussed earlier, we decompose problems such that we can describe them in programming code.  We can describe programming as the process of
breaking a large, complex task into smaller and smaller subtasks until the subtasks are simple enough to be performed with sequences of these basic
instructions.

#### Natural and Formal Languages

*Natural languages*  are the languages that people speak, such as English,
Spanish, and French. They were not designed by people (although people try to
impose some order on them); they evolved naturally.

*Formal languages*  are languages that are designed by people for specific
applications. For example, the notation that mathematicians use is a formal
language that is particularly good at denoting relationships among numbers and
symbols. Chemists use a formal language to represent the chemical structure of
molecules. And most importantly:

> Programming languages are formal languages that have been designed to express computations.

Formal languages tend to have strict rules about syntax. For example, $3*3=9$
is a syntactically correct mathematical statement, but $3=*9$ is not.
$H_{2}O$ is a syntactically correct chemical name, but $_{2}HO$ is
not.

Syntax rules come in two flavors, pertaining to *tokens* and *structure*.
Tokens are the basic elements of the language, such as words, numbers, parentheses,
commas, and so on.

In Python, a statement like `print("Happy New Year for ",2020)`
has 6 tokens: a function name, an open parenthesis (round bracket), a string, a comma, a number, and a close parenthesis.  If we were to type `write("Happy New Year for ",2020)` because the token `write` is meaningless in the Python formal language.  This is a great example of a syntax error.

The second type of syntax rule pertains to the *structure* of a statement---that is, the way the tokens are arranged. In Python example, if we omitted the comma, the interpreter would be confused by having a number come after the double quotes, resulting in illegal syntax.

When you read a sentence in English or a statement in a formal language, you
have to figure out what the structure of the sentence is (although in a natural
language you do this subconsciously). This process is called *parsing*.

For example, a common English phrase is to: "wait for the other shoe to drop".  This phrase can have a literal meaning, where some one is technically waiting for a shoe to fall to the floor due to gravity.  However, this is a rare usage for the phrase. More commonly, it is used to mean deferring an action until you understand the consequences.  Whatever the meaning, the phrase is parsed the same way in either case, however the *semantics* of the sentence are very different depending on the context.

In a similar way, there are differences to how various tokens in a programming language will be interpreted depending on the structure they are used in.  This is one of the complexity of programming languages, is that tokens are often used to mean multiple things depending on their context.  We will try to call these out throughout this volume at appropriate times.

Although formal and natural languages have many features in common --- tokens,
structure, syntax, and semantics --- there are many differences:

- **Ambiguity:** Natural languages are full of ambiguity, which people deal with by using contextual clues and other information. Formal languages are designed to be nearly or completely unambiguous, which means that any statement has exactly one meaning, regardless of context.
- **Redundancy:** In order to make up for ambiguity and reduce misunderstandings, natural languages employ lots of redundancy. As a result, they are often verbose.  Formal languages are less redundant and more concise.
- **Literalness:** Formal languages mean exactly what they say.  On the other hand, natural languages are full of idiom and metaphor. If someone says, "wait for the other shoe to drop", it seldom means that there is a shoe falling to the ground.

People who grow up speaking a natural language---everyone---often have a hard
time adjusting to formal languages. In some ways, the difference between formal
and natural language is like the difference between poetry and prose:

- **Poetry:** Words are used for their sounds as well as for their meaning, and the whole poem together creates an effect or emotional response. Ambiguity is not only common but often deliberate.
- **Prose:** The literal meaning of words is more important, and the structure contributes more meaning. Prose is more amenable to analysis than poetry but still often ambiguous.
- **Program:** The meaning of a computer program is unambiguous and literal, and can be understood entirely by analysis of the tokens and structure.

Here are some suggestions for reading programs (and other formal languages).
First, remember that formal languages are much more dense than natural
languages, so it takes longer to read them. Also, the structure is very
important, usually based on left-to-right, top-to-bottom reading conventions.  That being said, it is usually not a good idea to read from top to bottom, left to
right. Instead, learn to parse the program in your head, decompose the program into smaller parts until you can identify structures and tokens that have semantic meanings. Finally, the details matter. Little things
like spelling errors and bad punctuation, which you can get away with in
natural languages, can make a big difference in a formal language.  These are all immensely important to the process of debugging.

#### What is debugging?

Programming is a complex process, and because it is done by human beings, it

often leads to errors. Programming errors are called *bugs*  and the process of tracking them down and correcting them is called *debugging*.   (The common history of bugs in programming comes from Grace Hopper, one of the pioneers of computer science, and her team finding a moth getting stuck in the vacuum tubes of the computers they were working on previous to the invention of transistors.  This term then migrated to describing input/output errors within her team.

Some will note that the use of the term "bug" to describe small engineering difficulties dates back to at least 1889, when Thomas Edison had a bug with his phonograph and that "fly in the ointment" goes back many millennia, with the first known printing of the phrase being in John Norris' A *Practical Treatise Concerning Humility) in 1707. These stories detract from Hopper's place in history and her achievements in computing, which should not be underplayed or undervalued.*

Broadly, there are three kinds of errors can occur in a program:

- **Syntax Errors**  Unlike humans, who can quickly adopt new words (e.g. salutogenesis, qubit, gig economy), computers have a very set language that is immutable until the programming language is updated.  A syntax error occurs when the programmer puts in code that the interpreter does not recognize as being part of the language.
- **Semantic Error**  Sometimes, a program can run to completion without crashing, but still get the wrong answer.  This is a particular type of run time error where the *logic* the programmer has used is incorrect.  The computer executes it flawlessly, but the code itself is not correct.  A good example of a logic error is if you wanted to run through a list of students and determine if they pass a course. Students and teachers may say "You pass if your mark is above 50."  However, this is technically incorrect.  If the program code reads each grade and checks if a mark is *greater than* 50, it will lead to a different, and incorrect, result if used in place of checking if a mark is *greater than or equal* to 50.
- **Runtime Errors:**  A computer will only ever do what a program, and ultimately a programmer, tells it to do.  Sometimes, when we are writing code, we tell the computer to do the wrong thing, knowingly or unknowingly.  For example, we might ask the computer to process 100 numbers, and then hand it 101, meaning it will not know what to do with the last one.  This can sometimes result in a silent error ignoring the 101st number, or it may result in a crash when we tried to add the 101st number to a 100 item list.  Other examples of run time errors are when we try to divide by zero, access a file that is not available, or accessing some data that has never been defined as a variable (more on this in the next chapter).  Runtime errors are often referred to as *exceptions*.

Learning how to debug these errors takes time and practice.

#### Experimental debugging

One of the most important skills you will acquire is debugging.  Although it
can be frustrating, debugging is one of the most intellectually rich,
challenging, and interesting parts of programming.

In some ways, debugging is like detective work. You are confronted with clues,
and you have to infer the processes and events that led to the results you see.

Debugging is also like an experimental science. Once you have an idea what is
going wrong, you modify your program and try again. If your hypothesis was
correct, then you can predict the result of the modification, and you take a
step closer to a working program. If your hypothesis was wrong, you have to
come up with a new one. As Sherlock Holmes pointed out, When you have
eliminated the impossible, whatever remains, however improbable, must be the
truth. (A. Conan Doyle, *The Sign of Four*, 1890.)

For some people, programming and debugging are the same thing. That is,
programming is the process of gradually debugging a program until it does what
you want. The idea is that you should start with a program that does
*something* and make small modifications, debugging them as you go, so that you always have a working program.

For example, Linux is an operating system kernel that contains millions of
lines of code, but it started out as a simple program Linus Torvalds used to
explore the Intel 80386 chip. According to Larry Greenfield, one of Linus's
earlier projects was a program that would switch between displaying AAAA and
BBBB. This later evolved to Linux - a fully alternative and open source operating systems that is used all over the world.

Later in the course we will will make more suggestions about debugging and other programming practices.

#### The First Program

Traditionally, the first program written in a new language is called *Hello,
World!* because all it does is display the words, Hello, World!  In Python, the script looks like this:

In [ ]:
print("Hello, World!")

This is an example of using the *print function*, which doesn't actually print
anything on paper. It displays a value on the screen. In this case, the result shown
is

```

Hello World!

```

Note that the quotation marks in the program mark the beginning and end of the value do not appear in the result.  They are part of the syntax of the language!

Some people judge the quality of a programming language by the simplicity of
the Hello, World! program. By this standard, Python does about as well as
possible!

#### Comments

As programs get bigger and more complicated, they get more difficult to read.
Formal languages are dense, and it is often difficult to look at a piece of
code and figure out what it is doing, or why.

For this reason, it is a good idea to add notes to your programs to explain in
natural language what the program is doing.  These notes not only will help you in the future, but if anyone ever needs to build on top of your work in the future, then commenting will go a long way to helping them understand what you did --- even if you are no longer around to tell them!

A *comment* in a computer program is text that is intended
only for the human reader ---it is completely ignored by the interpreter.

In Python, the `#` token starts a comment.  The rest of the line
is ignored.   Here is a new version of *Hello, World!*.

In [ ]:
#---------------------------------------------------
# This demo program shows off Python!
# Written by Christopher Power, December 2020.
# Anyone may freely copy or modify this program.
#---------------------------------------------------
        
print("Hello, World!")     # This is my first program!

You'll also notice that we've left a blank line in the program.  Blank lines
are also ignored by the interpreter, but comments and blank lines can make your
programs much easier for humans to parse.  Use them liberally!